# ROUGE Evaluation: Condition A vs Condition B

## What this notebook does

This notebook evaluates the quality of the LLM-generated summaries using **ROUGE** metrics and answers the central research question of this project:

> **Does labelling each email with its speech act (Request, Propose, Commit…) before summarising produce better summaries than summarising the raw text directly?**

- **Condition A** — the LLM received the raw email thread text only.
- **Condition B** — the LLM received the same text, but each email was additionally annotated with speech act labels predicted by the BERT+LoRA classifier.

Both conditions used the same model (Gemini 3.6 Flash) and the same system instruction. The only difference was the presence of the labels in the prompt.

---

## What is ROUGE?

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is the standard automatic metric for summarisation. It measures **word overlap** between a generated summary and one or more human-written reference summaries. We report the **F1 score** (balances precision and recall) for three variants:

| Metric | What it measures |
|--------|------------------|
| **ROUGE-1** | Overlap of individual words (unigrams) |
| **ROUGE-2** | Overlap of two-word sequences (bigrams) — more sensitive to phrasing |
| **ROUGE-L** | Longest common subsequence — rewards preserving sentence-level order and flow |

Scores range from 0 (no overlap) to 1 (perfect overlap). In practice, abstractive summaries (paraphrased, not copy-pasted) typically score between 0.30–0.50 on ROUGE-1 against human references — lower than extractive systems, but expected.

---

## Reference summaries and the multi-annotator setup

The BC3 corpus provides **3 human-written reference summaries per thread** (one per annotator). These annotators worked independently, so their summaries differ in length, focus, and wording. Computing ROUGE against a single annotator would be unfair — we might score low simply because we matched annotator 1's phrasing but not annotator 2's.

**Solution:** we compute ROUGE against each of the 3 annotators separately and keep the **maximum score**. This is the standard approach used in DUC/TAC evaluations and in the original BC3 paper. It answers: *how close is the generated summary to the best possible human reference?*

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display, Markdown

here = Path(os.path.abspath(''))
REPO_ROOT = here.parent if here.name == 'notebooks' else here

scores = pd.read_csv(REPO_ROOT / 'results' / 'bc3_rouge_scores.csv')
gen_a  = pd.read_csv(REPO_ROOT / 'results' / 'bc3_summaries_condition_a.csv')
gen_b  = pd.read_csv(REPO_ROOT / 'results' / 'bc3_summaries_condition_b.csv')

print(f'Loaded scores for {len(scores)} threads')
print(scores[['listno', 'thread_name']].to_string(index=False))

## Per-thread ROUGE scores

In [ ]:
display_cols = {
    'thread_name': 'Thread',
    'rouge1_a': 'R1 (A)', 'rouge1_b': 'R1 (B)',
    'rouge2_a': 'R2 (A)', 'rouge2_b': 'R2 (B)',
    'rougeL_a': 'RL (A)', 'rougeL_b': 'RL (B)',
}
table = scores[list(display_cols.keys())].rename(columns=display_cols)

# Add a winner column per metric for quick reading
for m, label in [('rouge1', 'R1'), ('rouge2', 'R2'), ('rougeL', 'RL')]:
    table[f'{label} winner'] = scores.apply(
        lambda r: 'B' if r[f'{m}_b'] > r[f'{m}_a'] else ('A' if r[f'{m}_a'] > r[f'{m}_b'] else 'tie'),
        axis=1
    )

display(table.to_string(index=False))

## Global averages — Condition A vs B

In [ ]:
metrics = ['rouge1', 'rouge2', 'rougeL']
avg_a = [scores[f'{m}_a'].mean() for m in metrics]
avg_b = [scores[f'{m}_b'].mean() for m in metrics]

summary_df = pd.DataFrame({
    'Metric':      ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
    'Condition A': avg_a,
    'Condition B': avg_b,
    'Delta (B-A)': [b - a for a, b in zip(avg_a, avg_b)],
})
summary_df['Delta (B-A)'] = summary_df['Delta (B-A)'].map(lambda x: f'+{x:.4f}' if x >= 0 else f'{x:.4f}')
summary_df[['Condition A', 'Condition B']] = summary_df[['Condition A', 'Condition B']].round(4)

display(summary_df.to_string(index=False))

## Bar chart: ROUGE-1, ROUGE-2, ROUGE-L — A vs B

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
short_names = [n[:28] + '…' if len(n) > 28 else n for n in scores['thread_name']]
x = range(len(scores))
width = 0.35
colors = {'A': '#4878CF', 'B': '#D65F5F'}

for ax, metric, label in zip(axes, metrics, ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']):
    vals_a = scores[f'{metric}_a']
    vals_b = scores[f'{metric}_b']
    ax.bar([i - width/2 for i in x], vals_a, width, label='Condition A', color=colors['A'], alpha=0.85)
    ax.bar([i + width/2 for i in x], vals_b, width, label='Condition B', color=colors['B'], alpha=0.85)
    ax.axhline(vals_a.mean(), color=colors['A'], linestyle='--', linewidth=1.2, alpha=0.7)
    ax.axhline(vals_b.mean(), color=colors['B'], linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xticks(list(x))
    ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=7.5)
    ax.set_ylim(0, 0.55)
    ax.set_ylabel('F1 score')
    ax.legend(fontsize=8)

fig.suptitle('ROUGE scores per thread — Condition A (raw) vs Condition B (with labels)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'results' / 'bc3_rouge_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/bc3_rouge_comparison.png')

## Per-thread analysis — what does each summary look like?

In [ ]:
merged = gen_a.merge(gen_b[['listno', 'summary_b']], on='listno').merge(
    scores[['listno', 'rouge1_a', 'rouge1_b', 'rouge2_a', 'rouge2_b', 'rougeL_a', 'rougeL_b']], on='listno'
)

for _, row in merged.iterrows():
    r1_winner = 'B' if row['rouge1_b'] > row['rouge1_a'] else 'A'
    rL_winner = 'B' if row['rougeL_b'] > row['rougeL_a'] else 'A'
    display(Markdown(
        f"---\n"
        f"### `{row['listno']}` — {row['thread_name']}\n\n"
        f"| | ROUGE-1 | ROUGE-2 | ROUGE-L |\n"
        f"|---|---|---|---|\n"
        f"| **Cond. A** | {row['rouge1_a']:.4f} | {row['rouge2_a']:.4f} | {row['rougeL_a']:.4f} |\n"
        f"| **Cond. B** | {row['rouge1_b']:.4f} | {row['rouge2_b']:.4f} | {row['rougeL_b']:.4f} |\n\n"
        f"**ROUGE-1 winner:** Condition {r1_winner} &nbsp;&nbsp; **ROUGE-L winner:** Condition {rL_winner}\n\n"
        f"**Condition A summary:**\n\n{row['summary_a']}\n\n"
        f"**Condition B summary:**\n\n{row['summary_b']}"
    ))

## Interpretation and conclusions

### Overall picture

Across the 6 test threads, **Condition B (with speech act labels) outperforms Condition A on average for all three ROUGE metrics**:

| Metric | Cond. A | Cond. B | Improvement |
|--------|---------|---------|-------------|
| ROUGE-1 | 0.3928 | 0.3990 | +0.0062 |
| ROUGE-2 | 0.1027 | 0.1124 | +0.0097 |
| ROUGE-L | 0.2008 | 0.2239 | +0.0231 |

The improvement is most pronounced in **ROUGE-L (+0.023)**, which is the most meaningful metric for summarisation: it rewards preserving the logical flow and key information in the correct order — exactly what speech act labels help signal (e.g. knowing that an email is a *Commit* makes the model more likely to include that decision in the summary).

### Per-thread picture

The results are not uniform across threads. Condition B wins on ROUGE-L for **4 of the 6 threads**, but loses on 2:

- **059-11070771** (Phone connection to f2f): B wins clearly on all metrics — the thread has a mix of Requests and Meeting logistics that the labels help prioritise.
- **015-2625401** (SWADEurope postcard): A wins — this thread is very short (design feedback), where the labels add noise rather than structure.
- **061-10140940** (Non-geek guidelines): Mixed — A wins on ROUGE-1/2 but B wins on ROUGE-L.

### What the absolute scores tell us

ROUGE-1 scores around **0.39–0.40** are typical for **abstractive summarisation** against extractive-style human references. The model paraphrases rather than copies sentences from the thread, so exact word overlap is naturally lower than what an extractive system would score. This is expected and not a sign of poor quality — it simply means ROUGE has its limits as the sole evaluation criterion.

### Caveats and limitations

1. **Small test set (6 threads):** the differences between A and B are small in absolute terms and cannot be claimed as statistically significant with 6 data points. The direction of the effect (B ≥ A on average) is consistent with the hypothesis, but a larger test set would be needed to confirm it.
2. **ROUGE measures overlap, not quality:** a summary can be factually accurate and well-written while scoring low on ROUGE simply because it uses different vocabulary than the reference. The manual review below complements ROUGE with a qualitative lens.
3. **Label quality:** Condition B relies on BERT+LoRA predictions (Micro F1 = 0.637, not perfect). If the classifier mislabels an email, the label in the prompt may mislead the model rather than help it.

### Bottom line

The automatic evaluation provides **weak positive evidence** that speech act labels improve summarisation quality, with the clearest signal in ROUGE-L. The hypothesis is supported in direction but not magnitude. A richer evaluation (more threads, human judges) would be needed to draw stronger conclusions — which makes this a well-scoped research project rather than a conclusive one.